# Overview
Clustering patient data provides a powerful framework for uncovering latent structure in heart-transplant survival that may be obscured in traditional modeling approaches. By segmenting patients based on pre-transplant characteristics—including demographics, clinical history, hemodynamics, comorbidities, and laboratory values—we aim to identify clinically meaningful subgroups that differ in risk profiles, physiological presentation, and post-transplant outcomes.

While clustering is inherently exploratory rather than predictive, it plays a critical role in revealing population heterogeneity and identifying candidate phenotypes that may not be well captured through standard regression-based methods. This is particularly relevant when examining gender-specific survival patterns, where complex interactions among biological, clinical, and social factors may give rise to distinct patient subpopulations that are difficult to model parametrically.

In this study, we leverage data from the United Network for Organ Sharing (UNOS) covering the most recent 10-year period to ensure temporal consistency across clinical practice, allocation policies, and data completeness. This constrained time horizon reduces confounding from long-term systemic changes and enables more interpretable, era-specific characterization of both recipient and donor profiles. By jointly analyzing these features, we aim to better understand how patient-donor interactions, including gender related differences, contribute to variation in post-transplant survival outcomes.

In [1]:
# path to user functions
import sys  
sys.path.append('../Src/')

import pandas as pd
from pandas.api.types import CategoricalDtype
import numpy as np
from scipy import stats
import sys
from importlib.metadata import version

# initializing variables
SEED = 1776

# python modules
import utilities as u

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Scipy": sys.modules['scipy'].__version__,
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# adjust pandas display options to max
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
# adjust pandas display options to ensure full display of content
pd.set_option('display.max_colwidth', None)

  Library  Version
0  Python  3.11.13
1  Pandas    2.3.1
2   NumPy    2.3.2
3   Scipy   1.16.1


## Import Data

In [2]:
# import discretized data
df = pd.read_pickle("../Data/Heart_main.pkl")
df_dict =  pd.read_pickle("../Data/Dict_main.pkl")

In [3]:
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,INUTERO,Gender_CAN,BloodGroup_CAN,Weight_kg_Registrsation_CAN,Height_cm_Registration_CAN,BMI_Listing_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportInhaled_CAN,InotropesIVRegistration_CAN,LifeSupportRegistration_PGE_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceTypeRegistration_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,CerebroVascularDisease_CAN,PreviousMalignancy_CAN,CreatinineRegistration_CAN,TotalSerumAlbuminRegistration_CAN,DefibrillatorImplantRegistration_CAN,HemodynamicsRegistration_SYS_CAN,HemodynamicsRegistration_PA_DIA_CAN,HemodynamicsRegistration_PA_MN_CAN,HemodynamicsRegistration_PCW_CAN,HemodynamicsRegistration_CO_CAN,InotropesVasodilatorsRegistration_SYS_CAN,InotropesVasodilatorsRegistration_DIA_CAN,InotropesVasodilatorsRegistration_MN_CAN,InotropesVasodilatorsRegistration_PCW_CAN,InotropesVasodilatorsRegistration_CO_CAN,CigaretteUse_CAN,CigaretteAbstinence_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,PriorCardiacSurgeryTypeText_CAN,DAYS_STAT1,StatusDays_1A,StatusDays_2,StatusDays_1B,StatusDays_A4,StatusDays_A5,StatusDays_A2,StatusDays_A3,StatusDays_1,StatusDays_A6,LastInactiveStatusReason,InitialWaitingListStatusCode_CAN,ReasonRemovalWaitingList_CAN,ReceivedDeceasedDonorTramsplant_CAN,TotalDayWaitList_CAN,StatusAtTransplant_CAN,Age_Listing_CAN,LifeSupportRegistration_CAN,AllocationBeginDate_CAN,RemovalWaitListDate_CAN,InitialWaitListDate_CAN,Hispanic_CAN,Ethnicity_CAN,Height_cm_Listing_CAN,Weight_kg_Listing_CAN,BMI_Listing_CALC_CAN,Height_cm_Removal_CAN,Weight_kg_Removal_CAN,BMI_Removal_CAN,COMPOSITE_DEATH_DATE,VentilatorRegistration_CAN,TransplantRegion_CAN,ValidationDateTCR_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,STATUS_TRR,AdmissionDate_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,LifeSupportTransplant_PGE_CAN,CreatinineTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,HemodynamicsTransplant_CO_CAN,HemodynamicsTransplant_PA_DIA_CAN,HemodynamicsTransplant_PA_MN_CAN,HemodynamicsTransplant_PCW_CAN,HemodynamicsTransplant_SYS_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_MN_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,LifeSupportMechanismTransplant_OTHER_CAN,PriorLungSurgeryAfterRegistration_CAN,AirwayDehiscencePostTransplant,AcuteRejectionEpisode,StrokePostTransplant,DialysisPostDischarge,PacemakerPostTransplant,SteroidsUse_CAN,TotalBilirubinTransplant_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,LifeSupportInhaledTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,COD,GraftFailStatus,GraftLifeSpanDay,LastFollowupNumber,TransplantStatus,TransplantSurvivalDay,RecipientStatus,FunctionalStatusFollowUp,TXHRT,TransplantProcedure_CAN,STATUS_TCR,LifeSupportInhaledRegistration_CAN,DeceasedRetyped_DON,CrossMatchDone,CPRA_Recent_CAN,CPRA_Peak_CAN,RejectionTreatmentWithinOneYear,PreviousTransplantSameOrgan_CAN,PreviousT

In [4]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30725 entries, 0 to 30724
Data columns (total 295 columns):
 #    Column                                        Non-Null Count  Dtype         
---   ------                                        --------------  -----         
 0    PreviousTransplantNumber_CAN                  30725 non-null  category      
 1    WaitListDiagnosisCode_CAN                     30725 non-null  category      
 2    INUTERO                                       6319 non-null   object        
 3    Gender_CAN                                    30725 non-null  category      
 4    BloodGroup_CAN                                30725 non-null  category      
 5    Weight_kg_Registrsation_CAN                   30673 non-null  float64       
 6    Height_cm_Registration_CAN                    30608 non-null  float64       
 7    BMI_Listing_CAN                               30606 non-null  float64       
 8    Citizenship_CAN                               30725 no

In [5]:
df_dict.info(max_cols=df_dict.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 306 entries, 0 to 305
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Feature            306 non-null    object        
 1   Description        306 non-null    object        
 2   Form               304 non-null    object        
 3   FeatureStartDate   244 non-null    datetime64[ns]
 4   FeatureEndDate     9 non-null      datetime64[ns]
 5   FormSection        306 non-null    object        
 6   DataType           306 non-null    object        
 7   SASAnalysisFormat  306 non-null    object        
 8   Comment            306 non-null    object        
 9   OrginalFeature     306 non-null    object        
 10  FeatureType        306 non-null    object        
 11  Information        306 non-null    object        
dtypes: datetime64[ns](2), object(10)
memory usage: 28.8+ KB


## Data Wrangle (Complete Data)

#### User Function(s)

In [6]:
def get_cols_by_cardinality(data, cat, dropna=True, flag=False):
    if flag:
        # return columns with cardinality > cat
        return [
            col for col in data.columns
            if data[col].nunique(dropna=dropna) > cat
        ]
    else:
        # return columns with cardinality <= cat
        return [
            col for col in data.columns
            if data[col].nunique(dropna=dropna) <= cat
        ]


def get_column_summary(data, cat=2, flag=True, dropna=True, ignore_list=None):
    """
    Categorizes columns based on unique value counts.
    
    Args:
        data: DataFrame to analyze.
        cat: The threshold for the number of unique values.
        flag: If True, finds columns > cat. If False, finds columns <= cat.
        dropna: Whether to count NaNs as a unique value.
        ignore_list: List of column names to exclude from the result.
    """
    if ignore_list is None:
        ignore_list = []

    # Identify columns based on the 'cat' threshold
    if flag:
        cols = [col for col in data.columns if data[col].nunique(dropna=dropna) > cat]
    else:
        cols = [col for col in data.columns if data[col].nunique(dropna=dropna) <= cat]

    # Efficiently remove ignored columns
    cols = [col for col in cols if col not in ignore_list]
    
    # Create a dictionary of unique values
    summary = {col: data[col].unique().tolist() for col in cols}
    
    print(f"--- Found {len(cols)} Columns (Threshold: {'>' if flag else '<='} {cat}) ---")
    
    # Truncate long lists for cleaner printing
    for key, value in summary.items():
        val_str = f"{value[:5]}..." if len(value) > 5 else f"{value}"
        print(f"{key} : {val_str}")

    return list(summary.keys())

In [7]:
# remove cols
remove_cols = ['DAYS_STAT1',
 'StatusDays_1A',
 'StatusDays_1B',
 'StatusDays_2',
 'StatusDays_1',
 'StatusDays_A2',
 'StatusDays_A3',
 'StatusDays_A4',
 'StatusDays_A5',
 'StatusDays_A6',
 'LastInactiveStatusReason',
 'ReasonRemovalWaitingList_CAN',
 'STATUS_TRR',
 'STATUS_TCR',
 'STATUS_DDR',
 'AcuteRejectionEpisode',
 'AirwayDehiscencePostTransplant',
 'StrokePostTransplant',
 'PacemakerPostTransplant',
 'DialysisPostDischarge',
 'GraftFailStatus',
 'GraftLifeSpanDay',
 'LastFollowupNumber',
 'GraftStatus',
 'TransplantStatus',
 'RecipientStatus',
 'RejectionTreatmentWithinOneYear',
 'FunctionalStatusFollowUp',
 'LengthOfStay']
# remove unwanted features
df = df.drop(columns=remove_cols).copy()

In [8]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30725 entries, 0 to 30724
Data columns (total 266 columns):
 #    Column                                        Non-Null Count  Dtype         
---   ------                                        --------------  -----         
 0    PreviousTransplantNumber_CAN                  30725 non-null  category      
 1    WaitListDiagnosisCode_CAN                     30725 non-null  category      
 2    INUTERO                                       6319 non-null   object        
 3    Gender_CAN                                    30725 non-null  category      
 4    BloodGroup_CAN                                30725 non-null  category      
 5    Weight_kg_Registrsation_CAN                   30673 non-null  float64       
 6    Height_cm_Registration_CAN                    30608 non-null  float64       
 7    BMI_Listing_CAN                               30606 non-null  float64       
 8    Citizenship_CAN                               30725 no

* **TCR:** Transplant Candidate Registration
* **TRR:** Transplant Recipient Registration

In [9]:
u.any_nans(df)

--- Missing Values Found () (Total Rows: 30,725) ---
                                     Count Percentage
PriorCardiacSurgeryTypeText_CAN      24407   79.4369%
INUTERO                              24406   79.4337%
Class2PRA_TransplantPercentage_CAN   20895   68.0065%
Class1PRA_TransplantPercentage_CAN   20690   67.3393%
TotalSerumAlbuminRegistration_CAN    20652   67.2156%
CPRA_Peak_CAN                        15213   49.5134%
CPRA_Recent_CAN                      15201   49.4744%
OtherMedsText3_DON                   12763   41.5395%
ValidationDateTCR_CAN                 7761   25.2596%
OtherMedsText2_DON                    4300   13.9951%
HemodynamicsRegistration_PCW_CAN      2924    9.5167%
HemodynamicsTransplant_PCW_CAN        2601    8.4654%
HemodynamicsRegistration_CO_CAN       1801    5.8617%
HemodynamicsTransplant_CO_CAN         1740    5.6631%
HemodynamicsTransplant_PA_MN_CAN      1551    5.0480%
HemodynamicsRegistration_PA_MN_CAN    1451    4.7225%
HemodynamicsTransplant_PA_DIA

#### Features with > 40& Missingness & Dates REMOVED

In [10]:
# remove: include Hispanic_CAN & Date features (OtherMedsText3_DON keep since its part of two other fetures)
remove_cols = ['PriorCardiacSurgeryTypeText_CAN','INUTERO', 'Class2PRA_TransplantPercentage_CAN', 'Class1PRA_TransplantPercentage_CAN',
        'TotalSerumAlbuminRegistration_CAN', 'CPRA_Peak_CAN', 'CPRA_Recent_CAN', 'Hispanic_CAN']

# remove all dates
remove_cols.extend(df.columns[df.columns.str.contains('Date')].tolist())
# 
df = df.drop(columns=remove_cols).copy()
#
df.shape

(30725, 245)

In [11]:
df.head()

,PreviousTransplantNumber_CAN,WaitListDiagnosisCode_CAN,Gender_CAN,BloodGroup_CAN,Weight_kg_Registrsation_CAN,Height_cm_Registration_CAN,BMI_Listing_CAN,Citizenship_CAN,ResidencyStateRegistration_CAN,EducationLevel_CAN,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportInhaled_CAN,InotropesIVRegistration_CAN,LifeSupportRegistration_PGE_CAN,LifeSupportMechanismRegistration_OTHER_CAN,VentricularDeviceTypeRegistration_CAN,VentricularDeviceBrandRegistration_CAN,FunctionalStatusRegistration_CAN,PrimaryPaymentRegistration_CAN,DiagnosisAtListing_CAN,DiabetesType_CAN,DialysisTypeRegistration_CAN,CerebroVascularDisease_CAN,PreviousMalignancy_CAN,CreatinineRegistration_CAN,DefibrillatorImplantRegistration_CAN,HemodynamicsRegistration_SYS_CAN,HemodynamicsRegistration_PA_DIA_CAN,HemodynamicsRegistration_PA_MN_CAN,HemodynamicsRegistration_PCW_CAN,HemodynamicsRegistration_CO_CAN,InotropesVasodilatorsRegistration_SYS_CAN,InotropesVasodilatorsRegistration_DIA_CAN,InotropesVasodilatorsRegistration_MN_CAN,InotropesVasodilatorsRegistration_PCW_CAN,InotropesVasodilatorsRegistration_CO_CAN,CigaretteUse_CAN,CigaretteAbstinence_CAN,PriorCardiacSurgery_CAN,PriorCardiacSurgeryType_CAN,InitialWaitingListStatusCode_CAN,ReceivedDeceasedDonorTramsplant_CAN,TotalDayWaitList_CAN,StatusAtTransplant_CAN,Age_Listing_CAN,LifeSupportRegistration_CAN,Ethnicity_CAN,Height_cm_Listing_CAN,Weight_kg_Listing_CAN,BMI_Listing_CALC_CAN,Height_cm_Removal_CAN,Weight_kg_Removal_CAN,BMI_Removal_CAN,COMPOSITE_DEATH_DATE,VentilatorRegistration_CAN,TransplantRegion_CAN,WorkIncomeRegistration_CAN,AntigenBW4_CAN,AntigenBW6_CAN,AntigenC1_CAN,AntigenC2_CAN,AntigenDR51_CAN,AntigenDR51_2_CAN,AntigenDR52_CAN,AntigenDR52_2_CAN,AntigenDR53_CAN,AntigenDR53_2_CAN,AntigenDQ1_CAN,AntigenDQ2_CAN,FunctionalStatusTransplant_CAN,MedicalConditionTransplant_CAN,PrimaryPaymentTransplant_CAN,LifeSupportTransplant_ECMO_CAN,ResidencyStateTransplant_CAN,WorkIncomeTransplant_CAN,LifeSupportTransplant_PGE_CAN,CreatinineTransplant_CAN,DialysisBetweenRegistrationTransplant_CAN,HemodynamicsTransplant_CO_CAN,HemodynamicsTransplant_PA_DIA_CAN,HemodynamicsTransplant_PA_MN_CAN,HemodynamicsTransplant_PCW_CAN,HemodynamicsTransplant_SYS_CAN,LifeSupportTransplant_IABP_CAN,InfectionTherapyIV_CAN,InotropesIVTransplant_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_MN_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,LifeSupportMechanismTransplant_OTHER_CAN,PriorLungSurgeryAfterRegistration_CAN,SteroidsUse_CAN,TotalBilirubinTransplant_CAN,TransfusionAfterRegistration_CAN,VentricularDeviceTypeTransplant_CAN,VentricularDeviceBrandTransplant_CAN,VentilatorySupport_CAN,VentilatorTransplant_CAN,LifeSupportInhaledTransplant_CAN,PriorCardiacSurgeryTypeListAndTransplant_CAN,Hepatitis_B_CoreAntibody_CAN,SurfaceAntigenHEP_B_CAN,SurfaceHBVAntibodyTotalTransplant_CAN,CMVStatus_Transplant_CAN,HIV_SeroStatusTransplant_CAN,HEP_C_SerostatusStatus_CAN,EpsteinBarrSeroStatusTransplant_CAN,HIV_NAT_PreTransplant_CAN,HCV_NAT_PreTranspant_CAN,HBV_NAT_Result_CAN,COD,TransplantSurvivalDay,TXHRT,TransplantProcedure_CAN,LifeSupportInhaledRegistration_CAN,DeceasedRetyped_DON,CrossMatchDone,PreviousTransplantSameOrgan_CAN,PreviousTransplantAnyOrgan_CAN,AntigenDA1_DON,AntigenDA2_DON,AntigenDB1_DON,AntigenDB2_DON,AntigenDDR1_DON,AntigenDDR2_DON,AntigenRA1_CAN,AntigenRA2_CAN,AntigenRB1_CAN,AntigenRB2_CAN,AntigenRDR1_CAN,AntigenRDR2_CAN,MismatchLevel_AMIS,MismatchLevel_BMIS,MismatchLevel_DRMIS,MismatchLevel_HLAMIS,MalignancyBetweenRegistrationTransplant_CAN,CMV_IGG_Transplant_CAN,CMV_IGM_Transplant_CAN,Citizenship_DON,PastCocaineUse_DON,Age_DON,Ethnicity_DON,Hepatitis_B_CoreAntibody_DON,SurfaceAntigenHEP_B_DON,BloodGroup_DON,HeavyAlcoholUse_DON,DeceasedOrLiving_DON,Gender_DON,ResidencyState_DON,Antibody_HEP_C_DON,NonHeartBeating_DON,AntiHypertensive_DON,BloodInfectionSource_DON,BloodUreaNitrogenLevel_DON,Creatinine_DON,Othe

In [12]:
df.info(max_cols=df.shape[1])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30725 entries, 0 to 30724
Data columns (total 245 columns):
 #    Column                                        Non-Null Count  Dtype   
---   ------                                        --------------  -----   
 0    PreviousTransplantNumber_CAN                  30725 non-null  category
 1    WaitListDiagnosisCode_CAN                     30725 non-null  category
 2    Gender_CAN                                    30725 non-null  category
 3    BloodGroup_CAN                                30725 non-null  category
 4    Weight_kg_Registrsation_CAN                   30673 non-null  float64 
 5    Height_cm_Registration_CAN                    30608 non-null  float64 
 6    BMI_Listing_CAN                               30606 non-null  float64 
 7    Citizenship_CAN                               30725 non-null  category
 8    ResidencyStateRegistration_CAN                30725 non-null  category
 9    EducationLevel_CAN                   

In [22]:
df.columns[df.columns.str.contains('_Date').tolist()]

Index([], dtype='object')

In [13]:
cols = df.columns[~df.columns.str.endswith(('_DON','_CAN'))].tolist()
cols

['COMPOSITE_DEATH_DATE',
 'COD',
 'TransplantSurvivalDay',
 'TXHRT',
 'CrossMatchDone',
 'MismatchLevel_AMIS',
 'MismatchLevel_BMIS',
 'MismatchLevel_DRMIS',
 'MismatchLevel_HLAMIS',
 'BloodGroupMatchLevel']

In [15]:
df[cols].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
COMPOSITE_DEATH_DATE,30725.0,3069.0,1000.0,24367.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
COD,30725.0,73.0,1000.0,24487.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TransplantSurvivalDay,30300.0,NaN,NaN,NaN,1433.407195,1156.055867,0.0,369.0,1120.0,2207.0,4300.0
TXHRT,30725,1,Yes,30725,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CrossMatchDone,30725,3,Yes,28294,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_AMIS,30725,4,2,14469,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_BMIS,30725,4,2,20235,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_DRMIS,30725,4,2,15395,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MismatchLevel_HLAMIS,30725,8,5,10469,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BloodGroupMatchLevel,30725,3,Identical,26298,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# remove
cols.remove('TransplantSurvivalDay')
cols

['COMPOSITE_DEATH_DATE',
 'COD',
 'TXHRT',
 'CrossMatchDone',
 'MismatchLevel_AMIS',
 'MismatchLevel_BMIS',
 'MismatchLevel_DRMIS',
 'MismatchLevel_HLAMIS',
 'BloodGroupMatchLevel']

In [25]:
# remove unwanted features
df = df.drop(cols, axis=1).copy()

In [26]:
# sanity check
cols = df.columns[~df.columns.str.endswith(('_DON','_CAN'))].tolist()
cols

['TransplantSurvivalDay']

In [27]:
u.any_nans(df)

--- Missing Values Found () (Total Rows: 28,751) ---
                                     Count Percentage
OtherMedsText3_DON                   11804   41.0560%
OtherMedsText2_DON                    3859   13.4221%
HemodynamicsRegistration_PCW_CAN      2728    9.4884%
HemodynamicsTransplant_PCW_CAN        2425    8.4345%
HemodynamicsRegistration_CO_CAN       1659    5.7702%
HemodynamicsTransplant_CO_CAN         1609    5.5963%
HemodynamicsTransplant_PA_MN_CAN      1431    4.9772%
HemodynamicsRegistration_PA_MN_CAN    1325    4.6085%
HemodynamicsTransplant_PA_DIA_CAN     1169    4.0659%
HemodynamicsTransplant_SYS_CAN        1149    3.9964%
HemodynamicsRegistration_PA_DIA_CAN   1059    3.6834%
HemodynamicsRegistration_SYS_CAN      1038    3.6103%
OtherMedsText1_DON                     749    2.6051%
IschemicTimeHour_DON                   564    1.9617%
TotalBilirubinTransplant_CAN           462    1.6069%
TransplantSurvivalDay                  425    1.4782%
CreatinineTransplant_CAN     

In [28]:
# remove any NaNs for the label
df.dropna(subset=['TransplantSurvivalDay'], inplace=True)

### Ordinal

In [29]:
# ordinal features
ordinal_cols = ['PreviousTransplantNumber_CAN', 'EducationLevel_CAN', 'FunctionalStatusRegistration_CAN', 
                'FunctionalStatusTransplant_CAN', 'CigaretteAbstinence_CAN', 'MedicalConditionTransplant_CAN']

# display
df.loc[:, ordinal_cols].head()

,PreviousTransplantNumber_CAN,EducationLevel_CAN,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN,CigaretteAbstinence_CAN,MedicalConditionTransplant_CAN
0,0,ATTENDED COLLEGE/TECHNICAL SCHOOL,"10% - Moribund, fatal processes progressing rapidly","10% - Moribund, fatal processes progressing rapidly",Missing,In Intensive Care Unit
1,0,HIGH SCHOOL (9-12) or GED,"10% - Moribund, fatal processes progressing rapidly",90% - Able to carry on normal activity: minor symptoms of disease,Missing,Not Hospitalized
2,0,ATTENDED COLLEGE/TECHNICAL SCHOOL,"10% - Moribund, fatal processes progressing rapidly","20% - Very sick, hospitalization necessary: active treatment necessary",Missing,Hospitalized Not in ICU
3,0,HIGH SCHOOL (9-12) or GED,"20% - Very sick, hospitalization necessary: active treatment necessary",80% - Normal activity with effort: some symptoms of disease,0-2 months,Not Hospitalized
4,0,ASSOCIATE/BACHELOR DEGREE,90% - Able to carry on normal activity: minor symptoms of disease,90% - Able to carry on normal activity: minor symptoms of disease,Missing,Not Hospitalized


#### PreviousTransplantNumber_CAN

In [30]:
df.PreviousTransplantNumber_CAN.value_counts(dropna=False)

PreviousTransplantNumber_CAN
0    27401
1      875
2       45
3        5
Name: count, dtype: int64

In [31]:
df.PreviousTransplantNumber_CAN.dtype

CategoricalDtype(categories=[0, 1, 2, 3], ordered=False, categories_dtype=int64)

In [32]:
# Define the order
logical_order = [0, 1, 2, 3]

# Create the "Type" definition
cat_type = CategoricalDtype(categories=logical_order, ordered=True)

# Apply it directly to the column
# Note: Any value NOT in logical_order (like "Unknown") will become NaN
df['PreviousTransplantNumber_CAN'] = df['PreviousTransplantNumber_CAN'].astype(cat_type)
#
features = u.get_feature_info(df, 'PreviousTransplantNumber_CAN')

                              count  unique  top   freq
PreviousTransplantNumber_CAN  28326       4    0  27401

:::: NaN Count:
PreviousTransplantNumber_CAN    0 



#### EducationLevel_CAN

In [33]:
df.EducationLevel_CAN.value_counts(dropna=False)

EducationLevel_CAN
HIGH SCHOOL (9-12) or GED            10438
ATTENDED COLLEGE/TECHNICAL SCHOOL     7661
ASSOCIATE/BACHELOR DEGREE             5720
POST-COLLEGE GRADUATE DEGREE          2563
UNKNOWN                               1004
GRADE SCHOOL (0-8)                     869
NONE                                    49
Missing                                 22
Name: count, dtype: int64

In [34]:
# consolidate
df['EducationLevel_CAN'] = df['EducationLevel_CAN'].astype(str)   # convert to string
df.EducationLevel_CAN = df.EducationLevel_CAN.replace({"UNKNOWN":"Missing"})
df.EducationLevel_CAN.value_counts(dropna=False)

EducationLevel_CAN
HIGH SCHOOL (9-12) or GED            10438
ATTENDED COLLEGE/TECHNICAL SCHOOL     7661
ASSOCIATE/BACHELOR DEGREE             5720
POST-COLLEGE GRADUATE DEGREE          2563
Missing                               1026
GRADE SCHOOL (0-8)                     869
NONE                                    49
Name: count, dtype: int64

In [35]:
# verify Missing is Random
u.check_informative_missingness(df, 'EducationLevel_CAN', target='TransplantSurvivalDay', unknown_val='Missing')

--- Missingness (): EducationLevel_CAN ---
Group Sizes:    (Unknown=1,026, Known=27,300)
Mean survival:  (Unknown=1,414.8d, Known=1,326.9d)
Difference:     87.8 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0321
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     0.0826 (Negligible)
95% CI:        [0.0202, 0.1449]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



In [36]:
# Define the order
logical_order = ['NONE', 'GRADE SCHOOL (0-8)', 'HIGH SCHOOL (9-12) or GED', 'ATTENDED COLLEGE/TECHNICAL SCHOOL', 
                 'ASSOCIATE/BACHELOR DEGREE', 'POST-COLLEGE GRADUATE DEGREE', 'Missing']

# Create the "Type" definition
cat_type = CategoricalDtype(categories=logical_order, ordered=True)

# Apply it directly to the column
# Note: Any value NOT in logical_order (like "Unknown") will become NaN
df['EducationLevel_CAN'] = df['EducationLevel_CAN'].astype(cat_type)
#
features = u.get_feature_info(df, 'EducationLevel_CAN')

                    count unique                        top   freq
EducationLevel_CAN  28326      7  HIGH SCHOOL (9-12) or GED  10438

:::: NaN Count:
EducationLevel_CAN    0 



In [37]:
df.EducationLevel_CAN.dtype

CategoricalDtype(categories=['NONE', 'GRADE SCHOOL (0-8)', 'HIGH SCHOOL (9-12) or GED',
                  'ATTENDED COLLEGE/TECHNICAL SCHOOL',
                  'ASSOCIATE/BACHELOR DEGREE', 'POST-COLLEGE GRADUATE DEGREE',
                  'Missing'],
, ordered=True, categories_dtype=object)

#### FunctionalStatus

In [38]:
df.FunctionalStatusRegistration_CAN.value_counts(dropna=False)

FunctionalStatusRegistration_CAN
20% - Very sick, hospitalization necessary: active treatment necessary                                 6424
70% - Cares for self: unable to carry on normal activity or active work                                3976
40% - Disabled: requires special care and assistance                                                   3921
60% - Requires occasional assistance but is able to care for needs                                     3475
50% - Requires considerable assistance and frequent medical care                                       3020
30% - Severely disabled: hospitalization is indicated, death not imminent                              2473
80% - Normal activity with effort: some symptoms of disease                                            2462
Unknown                                                                                                 944
90% - Able to carry on normal activity: minor symptoms of disease                                      

In [39]:
mask = (
    ~df['FunctionalStatusRegistration_CAN'].astype(str).str.contains("%", na=False)
    & (df['FunctionalStatusRegistration_CAN'] != "Unknown")
)

# display
df.loc[mask, ['FunctionalStatusRegistration_CAN', 'FunctionalStatusTransplant_CAN']]

,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN
3995,2,80% - Normal activity with effort: some symptoms of disease
14942,2,"20% - Very sick, hospitalization necessary: active treatment necessary"
25163,2,40% - Disabled: requires special care and assistance
25801,2,"20% - Very sick, hospitalization necessary: active treatment necessary"
26722,2,"20% - Very sick, hospitalization necessary: active treatment necessary"
27350,1,60% - Requires occasional assistance but is able to care for needs
27461,1,70% - Cares for self: unable to carry on normal activity or active work
28344,1,"20% - Very sick, hospitalization necessary: active treatment necessary"


In [40]:
# regex: 
# Ignore any leading spaces: \s*
# Capture the number just before %: (\d+(?:\.\d+)?)
# Ignore spaces before the %: \s*
# Look ahead to ensure the next character is %  (but don’t consume it): (?=%)
df["FunctionalStatusRegistration_CAN"] = (
    df["FunctionalStatusRegistration_CAN"]
    .str.extract(r"\s*(\d+(?:\.\d+)?)\s*(?=%)", expand=False)
    .astype(float)
    / 100
)
df["FunctionalStatusTransplant_CAN"] = (
    df["FunctionalStatusTransplant_CAN"]
    .str.extract(r"\s*(\d+(?:\.\d+)?)\s*(?=%)", expand=False)
    .astype(float)
    / 100
)
#
features = u.get_feature_info(df, 'FunctionalStatus')

                                    count     mean       std  min  25%  50%  75%  max
FunctionalStatusRegistration_CAN  27374.0  0.47022  0.223377  0.1  0.2  0.5  0.7  1.0
FunctionalStatusTransplant_CAN    26974.0  0.43341  0.236474  0.1  0.2  0.4  0.6  1.0

:::: NaN Count:
FunctionalStatusRegistration_CAN     952
FunctionalStatusTransplant_CAN      1352 



In [41]:
# verify Missing is Random
u.check_informative_missingness(df, 'FunctionalStatusRegistration_CAN', target='TransplantSurvivalDay', unknown_val=None)

--- Missingness (): FunctionalStatusRegistration_CAN ---
Group Sizes:    (Unknown=952, Known=27,374)
Mean survival:  (Unknown=1,146.6d, Known=1,336.5d)
Difference:     -189.9 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -0.1786 (Negligible)
95% CI:        [-0.2432, -0.1139]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



In [42]:
# verify Missing is Random
u.check_informative_missingness(df, 'FunctionalStatusTransplant_CAN', target='TransplantSurvivalDay', unknown_val=None)

--- Missingness (): FunctionalStatusTransplant_CAN ---
Group Sizes:    (Unknown=1,352, Known=26,974)
Mean survival:  (Unknown=1,103.0d, Known=1,341.5d)
Difference:     -238.5 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -0.2245 (Small)
95% CI:        [-0.2791, -0.1698]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



#### CigaretteAbstinence_CAN

In [43]:
df.CigaretteAbstinence_CAN.value_counts(dropna=False)

CigaretteAbstinence_CAN
Missing               15899
>60 months             7076
3-12 months            2067
13-24 months            973
25-36 months            601
37-48 months            473
Unknown duration        447
49-60 months            413
0-2 months              324
Continues to smoke       53
Name: count, dtype: int64

In [44]:
# verify Missing is Random
u.check_informative_missingness(df, 'CigaretteAbstinence_CAN', target='TransplantSurvivalDay', unknown_val='Missing')

--- Missingness (): CigaretteAbstinence_CAN ---
Group Sizes:    (Unknown=15,899, Known=12,427)
Mean survival:  (Unknown=1,316.3d, Known=1,347.8d)
Difference:     -31.4 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0136
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -0.0295 (Negligible)
95% CI:        [-0.0530, -0.0061]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



#### MedicalConditionTransplant_CAN

In [45]:
df.MedicalConditionTransplant_CAN.value_counts(dropna=False)

MedicalConditionTransplant_CAN
Not Hospitalized           13362
In Intensive Care Unit     10625
Hospitalized Not in ICU     4294
Missing                       45
Name: count, dtype: int64

In [46]:
# verify Missing is Random
u.check_informative_missingness(df, 'MedicalConditionTransplant_CAN', target='TransplantSurvivalDay', unknown_val='Missing')

--- Missingness (): MedicalConditionTransplant_CAN ---
Group Sizes:    (Unknown=45, Known=28,281)
Mean survival:  (Unknown=102.4d, Known=1,332.1d)
Difference:     -1,229.7 days

--- Welch's t-test Analysis ---
ρ-Value: 0.0000
RESULT: Missingness is associated with survival (informative missingness).
Interpretation: Not MCAR. Likely MAR or MNAR.

--- Cohen's Analysis ---
Cohen's d:     -1.1571 (Large)
95% CI:        [-1.4497, -0.8645]
Result: INFORMATIVE MISSINGNESS (statistically significant difference; small effect size)



In [47]:
# Define the order
logical_order = ['Not Hospitalized', 'Hospitalized Not in ICU', 'In Intensive Care Unit', 'Missing']

# update column
df['MedicalConditionTransplant_CAN'] = df['MedicalConditionTransplant_CAN'].astype('string')

# Create the "Type" definition
cat_type = CategoricalDtype(categories=logical_order, ordered=True)

# Apply it directly to the column
# Note: Any value NOT in logical_order (like "Unknown") will become NaN
df['MedicalConditionTransplant_CAN'] = df['MedicalConditionTransplant_CAN'].astype(cat_type)
#
features = u.get_feature_info(df, 'MedicalConditionTransplant_CAN')

                                count unique               top   freq
MedicalConditionTransplant_CAN  28326      4  Not Hospitalized  13362

:::: NaN Count:
MedicalConditionTransplant_CAN    0 



In [48]:
df.MedicalConditionTransplant_CAN.dtype

CategoricalDtype(categories=['Not Hospitalized', 'Hospitalized Not in ICU',
                  'In Intensive Care Unit', 'Missing'],
, ordered=True, categories_dtype=object)

In [49]:
df.loc[:, ordinal_cols].sample(n=5, random_state=SEED)

,PreviousTransplantNumber_CAN,EducationLevel_CAN,FunctionalStatusRegistration_CAN,FunctionalStatusTransplant_CAN,CigaretteAbstinence_CAN,MedicalConditionTransplant_CAN
4570,0,HIGH SCHOOL (9-12) or GED,0.3,0.7,13-24 months,Not Hospitalized
20866,0,ASSOCIATE/BACHELOR DEGREE,0.6,0.8,>60 months,Not Hospitalized
23163,0,ASSOCIATE/BACHELOR DEGREE,0.3,0.2,Missing,Hospitalized Not in ICU
17289,0,HIGH SCHOOL (9-12) or GED,0.9,0.8,>60 months,Not Hospitalized
18059,0,ASSOCIATE/BACHELOR DEGREE,0.2,0.2,Missing,In Intensive Care Unit


### Binary Columns

In [50]:
# Ignore columns
ignore_cols = ['Gender_CAN', 'Gender_DON', 'TransplantType_CAN']
ignore_cols.extend(ordinal_cols)
# get binary features
binary_cols = get_column_summary(df, cat = 2, flag=False, ignore_list=ignore_cols)

--- Found 21 Columns (Threshold: <= 2) ---
LifeSupportRegistration_ECMO_CAN : [0, 1]
LifeSupportRegistration_IABP_CAN : [0, 1]
LifeSupportInhaled_CAN : [0, 1]
InotropesIVRegistration_CAN : [0, 1]
LifeSupportRegistration_PGE_CAN : [0, 1]
LifeSupportMechanismRegistration_OTHER_CAN : [1, 0]
ReceivedDeceasedDonorTramsplant_CAN : [1, 0]
VentilatorRegistration_CAN : ['No', 'Yes']
LifeSupportTransplant_ECMO_CAN : [0, 1]
LifeSupportTransplant_PGE_CAN : [0, 1]
LifeSupportTransplant_IABP_CAN : [0, 1]
InotropesIVTransplant_CAN : [0, 1]
LifeSupportMechanismTransplant_OTHER_CAN : [1, 0]
VentilatorTransplant_CAN : ['No', 'Yes']
LifeSupportInhaledTransplant_CAN : [0, 1]
TransplantProcedure_CAN : ['Heart']
LifeSupportInhaledRegistration_CAN : [0, 1]
PreviousTransplantSameOrgan_CAN : ['No', 'Yes']
PreviousTransplantAnyOrgan_CAN : ['No', 'Yes']
DeceasedOrLiving_DON : ['Deceased Donor', 'Living Donor']
LT_ONE_WEEK_DON : ['N']


In [51]:
df.DeceasedOrLiving_DON.value_counts()

DeceasedOrLiving_DON
Deceased Donor    28323
Living Donor          3
Name: count, dtype: int64

In [52]:
df.TransplantProcedure_CAN.value_counts()

TransplantProcedure_CAN
Heart    28326
Name: count, dtype: int64

In [53]:
# no value added
df = df.drop(['DeceasedOrLiving_DON', 'TransplantProcedure_CAN'], axis=1)
binary_cols.remove('DeceasedOrLiving_DON')
binary_cols.remove('TransplantProcedure_CAN')

In [54]:
# convert all binary types to boolean
mapping = {
    "Y": "Yes",
    "N": "No",
    "1": "Yes",
    "0": "No"
}

# Convert only selected columns
for col in binary_cols:
    df[col] = df[col].astype('str')
    df[col] = df[col].map(mapping).fillna(df[col])
    df[col] = df[col].astype('category')

# display
df.loc[:, binary_cols].head()

,LifeSupportRegistration_ECMO_CAN,LifeSupportRegistration_IABP_CAN,LifeSupportInhaled_CAN,InotropesIVRegistration_CAN,LifeSupportRegistration_PGE_CAN,LifeSupportMechanismRegistration_OTHER_CAN,ReceivedDeceasedDonorTramsplant_CAN,VentilatorRegistration_CAN,LifeSupportTransplant_ECMO_CAN,LifeSupportTransplant_PGE_CAN,LifeSupportTransplant_IABP_CAN,InotropesIVTransplant_CAN,LifeSupportMechanismTransplant_OTHER_CAN,VentilatorTransplant_CAN,LifeSupportInhaledTransplant_CAN,LifeSupportInhaledRegistration_CAN,PreviousTransplantSameOrgan_CAN,PreviousTransplantAnyOrgan_CAN,LT_ONE_WEEK_DON
0,No,No,No,No,No,Yes,Yes,No,No,No,No,No,Yes,No,No,No,No,No,No
1,No,No,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No
2,Yes,Yes,No,No,No,No,Yes,Yes,No,No,No,Yes,No,No,No,No,No,No,No
3,No,No,No,Yes,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No
4,No,No,No,No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No


In [55]:
# Ignore columns
ignore_cols.extend(binary_cols)
# get nonbinary features
binary_nan_cols = get_column_summary(df, cat = 3, flag=False, ignore_list=ignore_cols)

--- Found 25 Columns (Threshold: <= 3) ---
InotropesVasodilatorsRegistration_SYS_CAN : ['Yes', 'No', 'Missing']
InotropesVasodilatorsRegistration_DIA_CAN : ['Yes', 'No', 'Missing']
InotropesVasodilatorsRegistration_MN_CAN : ['Yes', 'No', 'Missing']
InotropesVasodilatorsRegistration_PCW_CAN : ['Yes', 'No', 'Missing']
InotropesVasodilatorsRegistration_CO_CAN : ['Yes', 'No', 'Missing']
CigaretteUse_CAN : ['No', 'Yes', 'Missing']
LifeSupportRegistration_CAN : ['Yes', 'No', 'Missing']
InotropesVasodilatorsTransplant_CO_CAN : ['Missing', 'No', 'Yes']
InotropesVasodilatorsTransplant_DIA_CAN : ['Missing', 'No', 'Yes']
InotropesVasodilatorsTransplant_MN_CAN : ['Missing', 'No', 'Yes']
InotropesVasodilatorsTransplant_PCW_CAN : ['Missing', 'No', 'Yes']
InotropesVasodilatorsTransplant_SYS_CAN : ['Missing', 'No', 'Yes']
DeceasedRetyped_DON : ['No', 'Yes', 'Missing']
NonHeartBeating_DON : ['No', 'Missing', 'Yes']
BloodInfectionSource_DON : [0, 1, 999]
OtherInfectionSource_DON : [0, 999, 1]
PulmonaryI

In [56]:
# convert all binary types to boolean
mapping = {
    '1': "Yes",
    '0': "No",
    '999': "Missing"
}

# Convert only selected columns
for col in binary_nan_cols:
    df[col] = df[col].astype('str')
    df[col] = df[col].map(mapping).fillna(df[col])
    df[col] = df[col].astype('category')

# display
df.loc[:, binary_nan_cols].head()

,InotropesVasodilatorsRegistration_SYS_CAN,InotropesVasodilatorsRegistration_DIA_CAN,InotropesVasodilatorsRegistration_MN_CAN,InotropesVasodilatorsRegistration_PCW_CAN,InotropesVasodilatorsRegistration_CO_CAN,CigaretteUse_CAN,LifeSupportRegistration_CAN,InotropesVasodilatorsTransplant_CO_CAN,InotropesVasodilatorsTransplant_DIA_CAN,InotropesVasodilatorsTransplant_MN_CAN,InotropesVasodilatorsTransplant_PCW_CAN,InotropesVasodilatorsTransplant_SYS_CAN,DeceasedRetyped_DON,NonHeartBeating_DON,BloodInfectionSource_DON,OtherInfectionSource_DON,PulmonaryInfection_DON,UrineInfection_DON,DialysisPriorRegistration_CAN,LifeSupportTransplant_CAN,CardiacArrest_DON,LV_EjectionFractionMedthod_DON,KidneyAllocation_DON,LungPO2_Done_DON,PulmonaryCatheter_DON
0,Yes,Yes,Yes,Yes,Yes,No,Yes,Missing,Missing,Missing,Missing,Missing,No,No,No,No,Yes,No,No,Yes,No,Echo,No,Yes,No
1,No,No,No,No,No,No,Yes,No,No,No,No,No,Yes,No,No,No,Yes,No,No,Yes,No,Echo,No,Yes,No
2,Yes,Yes,Yes,Missing,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,No,No,No,No,Yes,No,No,Yes,No,Echo,No,Yes,No
3,Missing,Missing,Missing,Missing,Missing,Yes,Yes,No,No,No,No,No,Yes,No,Yes,No,Yes,Yes,No,Yes,No,Echo,No,Yes,No
4,No,No,No,No,No,No,Yes,No,No,No,No,No,Yes,No,No,No,Yes,No,No,Yes,Yes,Echo,No,Yes,No


In [57]:
# Ignore columns
ignore_cols.extend(binary_nan_cols)
# get nonbinary features
non_binary_cols = get_column_summary(df, cat = 4, flag=False, ignore_list=ignore_cols)

--- Found 50 Columns (Threshold: <= 4) ---
CerebroVascularDisease_CAN : ['No', 'Yes', 'Unknown', 'Missing']
PreviousMalignancy_CAN : ['No', 'Yes', 'Unknown', 'Missing']
DefibrillatorImplantRegistration_CAN : ['Yes', 'No', 'Unknown', 'Missing']
PriorCardiacSurgery_CAN : ['Yes', 'No', 'Unknown', 'Missing']
WorkIncomeRegistration_CAN : ['No', 'Yes', 'Missing', 'Unknown']
AntigenBW4_CAN : ['0', 'Negative', 'Positive', 'Not Done']
AntigenBW6_CAN : ['0', 'Positive', 'Negative', 'Not Done']
WorkIncomeTransplant_CAN : ['No', 'Unknown', 'Yes', 'Missing']
DialysisBetweenRegistrationTransplant_CAN : ['No', 'Yes', 'Unknown', 'Missing']
InfectionTherapyIV_CAN : ['No', 'Unknown', 'Yes', 'Missing']
PriorLungSurgeryAfterRegistration_CAN : ['No', 'Yes', 'Unknown', 'Missing']
SteroidsUse_CAN : ['No', 'Yes', 'Unknown', 'Missing']
TransfusionAfterRegistration_CAN : ['No', 'Yes', 'Unknown', 'Missing']
VentilatorySupport_CAN : ['Yes', 'No', 'Unknown', 'Missing']
MalignancyBetweenRegistrationTransplant_CAN :

### Candidate

In [58]:
# keep candidate and TransplantSurvivalDay features
cols = df.columns[df.columns.str.contains("CAN")].to_list()
cols.extend(['TransplantSurvivalDay'])

# all the candidates' features and TransplantSurvivalDay
df_can = df[cols]

# display
df_can.shape

(28326, 129)

### Donor

In [59]:
# keep donor and TransplantSurvivalDay features
cols = df.columns[df.columns.str.contains("DON")].to_list()
cols.extend(['TransplantSurvivalDay'])

# all the candidates' features and TransplantSurvivalDay
df_don = df[cols]

# display
df_don.shape

(28326, 94)

In [60]:
# sanity check
df.shape[1], df_can.shape[1] + df_don.shape[1]

(222, 223)

#### Write to Disk: Full Data

In [61]:
# save data: heart dataset
u.write_to_file(df_don, 'DON_Heart_Data',path='../Data/', format='pkl')
u.write_to_file(df_can, 'CAN_Heart_Data',path='../Data/', format='pkl')

28,326 records written to ../Data/DON_Heart_Data.pkl
28,326 records written to ../Data/CAN_Heart_Data.pkl
